# Lezione 9 · Gli LLM: token, pretraining, scaling

L'esercizio dell'archivio, il taglio delle parole con il BPE, la curva della vasca, il duello delle due navi e il gradino della misura.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## L'esercizio dell'archivio

*Nel racconto: «copri con la mano la riga successiva e prova a indovinarla».*

In [2]:
controlla("se al token vero dava 0,5, paga", -math.log(0.5), 0.69)
controlla("se gli dava 0,01, paga", -math.log(0.01), 4.6)
controlla("una loss di 2: incerto come tra quanti token", math.exp(2), 7.4, 0.05)

✓  se al token vero dava 0,5, paga: calcolato 0.6931, nel libro 0.69
✓  se gli dava 0,01, paga: calcolato 4.605, nel libro 4.6
✓  una loss di 2: incerto come tra quanti token: calcolato 7.389, nel libro 7.4


## Il taglio delle parole

*Nel racconto: le bandiere di segnalazione; «conto le coppie di lettere vicine e fondo la più frequente».*

L'idea dell'allievo, in codice. È il primo kata per casa: il testo è «rete rete rete rete reti reti reti remi remi», e le fusioni non scavalcano le parole.

In [3]:
from collections import Counter

def conta_coppie(parole):
    coppie = Counter()
    for pezzi, volte in parole.items():
        for a, b in zip(pezzi, pezzi[1:]):
            coppie[(a, b)] += volte
    return coppie

def fondi(parole, coppia):
    nuove = {}
    for pezzi, volte in parole.items():
        fusi, i = [], 0
        while i < len(pezzi):
            if i + 1 < len(pezzi) and (pezzi[i], pezzi[i + 1]) == coppia:
                fusi.append(pezzi[i] + pezzi[i + 1]); i += 2
            else:
                fusi.append(pezzi[i]); i += 1
        nuove[tuple(fusi)] = volte
    return nuove

quanti = lambda parole: sum(len(pezzi) * volte for pezzi, volte in parole.items())
testo = "rete rete rete rete reti reti reti remi remi".split()
parole = {tuple(p): n for p, n in Counter(testo).items()}       # si parte dalle lettere
conta("token all'inizio", quanti(parole), 36)
for fusione, (pezzo, volte, dopo) in enumerate([("re", 9, 27), ("ret", 7, 20), ("rete", 4, 16)], 1):
    coppia, n = conta_coppie(parole).most_common(1)[0]
    parole = fondi(parole, coppia)
    assert "".join(coppia) == pezzo
    conta(f"fusione {fusione}: {coppia[0]} + {coppia[1]} → {pezzo}, quante volte compare", n, volte)
    conta(f"   token dopo la fusione", quanti(parole), dopo)
print("alla fine:", ",  ".join(" · ".join(p) for p in parole))
prossima, n = conta_coppie(parole).most_common(1)[0]
assert prossima == ("ret", "i")
conta("la fusione successiva sarebbe ret + i, che compare", n, 3)

✓  token all'inizio: calcolato 36, nel libro 36
✓  fusione 1: r + e → re, quante volte compare: calcolato 9, nel libro 9
✓     token dopo la fusione: calcolato 27, nel libro 27
✓  fusione 2: re + t → ret, quante volte compare: calcolato 7, nel libro 7
✓     token dopo la fusione: calcolato 20, nel libro 20
✓  fusione 3: ret + e → rete, quante volte compare: calcolato 4, nel libro 4
✓     token dopo la fusione: calcolato 16, nel libro 16
alla fine: rete,  ret · i,  re · m · i
✓  la fusione successiva sarebbe ret + i, che compare: calcolato 3, nel libro 3


In [4]:
# le lettere che il modello non vede
conta("le «r» di «ormeggiatori»", "ormeggiatori".count("r"), 2)
assert "".join(["orm", "eggi", "atori"]) == "ormeggiatori"

✓  le «r» di «ormeggiatori»: calcolato 2, nel libro 2


## La curva della vasca

*Nel racconto: i tre modellini, la curva che scende sempre meno e si appiattisce sopra una linea.*

La formula dello studio del 2022, con i coefficienti riportati nel libro, per modelli allenati con 20 token per parametro.

In [5]:
E, A, B, alfa, beta = 1.69, 406.4, 410.7, 0.34, 0.28
loss = lambda N, D: E + A / N ** alfa + B / D ** beta
calcolo = lambda N, D: 6 * N * D

sopra_il_fondo = []
for N, calcolo_atteso, loss_attesa in [(1e8, 1.2e18, 3.49), (1e9, 1.2e20, 2.58), (1e10, 1.2e22, 2.13), (1e11, 1.2e24, 1.91), (1e12, 1.2e26, 1.80)]:
    D = 20 * N
    assert math.isclose(calcolo(N, D), calcolo_atteso)
    controlla(f"{N / 1e9:g} miliardi di parametri, {D / 1e9:g} miliardi di token: loss prevista", loss(N, D), loss_attesa, 0.006)
    sopra_il_fondo.append(loss(N, D) - E)
for v, a in zip(sopra_il_fondo, [1.80, 0.89, 0.44, 0.22, 0.11]):
    controlla("la parte sopra il fondo", v, a, 0.006)
print("   ogni cento volte di calcolo si dimezza:", ", ".join(f"{b / a:.2f}" for a, b in zip(sopra_il_fondo, sopra_il_fondo[1:])))

✓  0.1 miliardi di parametri, 2 miliardi di token: loss prevista: calcolato 3.486, nel libro 3.49
✓  1 miliardi di parametri, 20 miliardi di token: loss prevista: calcolato 2.58, nel libro 2.58
✓  10 miliardi di parametri, 200 miliardi di token: loss prevista: calcolato 2.133, nel libro 2.13
✓  100 miliardi di parametri, 2000 miliardi di token: loss prevista: calcolato 1.912, nel libro 1.91
✓  1000 miliardi di parametri, 20000 miliardi di token: loss prevista: calcolato 1.801, nel libro 1.8
✓  la parte sopra il fondo: calcolato 1.796, nel libro 1.8
✓  la parte sopra il fondo: calcolato 0.89, nel libro 0.89
✓  la parte sopra il fondo: calcolato 0.4431, nel libro 0.44
✓  la parte sopra il fondo: calcolato 0.2216, nel libro 0.22
✓  la parte sopra il fondo: calcolato 0.1113, nel libro 0.11
   ogni cento volte di calcolo si dimezza: 0.50, 0.50, 0.50, 0.50


## Il duello delle due navi

*Nel racconto: «la prima è grandissima ma ha letto poco; la seconda è quattro volte più piccola».*

In [6]:
gopher = (280e9, 300e9)
chinchilla = (70e9, 1.4e12)
controlla("quante volte più piccola", gopher[0] / chinchilla[0], 4)
controlla("quante volte ha letto di più («quasi cinque»)", chinchilla[1] / gopher[1], 4.67)
controlla("calcolo di Gopher (× 10²³)", calcolo(*gopher) / 1e23, 5.0, 0.05)
controlla("calcolo di Chinchilla (× 10²³)", calcolo(*chinchilla) / 1e23, 5.9, 0.05)
controlla("token per parametro di Chinchilla", chinchilla[1] / chinchilla[0], 20)
print(f"   la formula della vasca dà {loss(*gopher):.2f} a Gopher e {loss(*chinchilla):.2f} a Chinchilla: vince la piccola")
assert loss(*chinchilla) < loss(*gopher)

✓  quante volte più piccola: calcolato 4, nel libro 4
✓  quante volte ha letto di più («quasi cinque»): calcolato 4.667, nel libro 4.67
✓  calcolo di Gopher (× 10²³): calcolato 5.04, nel libro 5
✓  calcolo di Chinchilla (× 10²³): calcolato 5.88, nel libro 5.9
✓  token per parametro di Chinchilla: calcolato 20, nel libro 20
   la formula della vasca dà 1.99 a Gopher e 1.94 a Chinchilla: vince la piccola


## Oltre la regola

*Nel racconto: «modelli piccoli che leggono cento volte di più».*

In [7]:
N, D = 8e9, 15e12
controlla("token per parametro", D / N, 1900, 30)
controlla("volte la regola («quasi cento»)", D / N / 20, 94, 0.5)
controlla("8 miliardi: il termine dei parametri", A / N ** alfa, 0.175, 0.001)
controlla("8 miliardi: il termine dei token", B / D ** beta, 0.084, 0.001)
controlla("8 miliardi: loss", loss(N, D), 1.95)
controlla("Chinchilla: il termine dei parametri", A / chinchilla[0] ** alfa, 0.083, 0.001)
controlla("Chinchilla: il termine dei token", B / chinchilla[1] ** beta, 0.163, 0.001)
controlla("Chinchilla: loss", loss(*chinchilla), 1.94)
controlla("quante volte più piccolo («quasi nove»)", chinchilla[0] / N, 8.75)

✓  token per parametro: calcolato 1875, nel libro 1900
✓  volte la regola («quasi cento»): calcolato 93.75, nel libro 94
✓  8 miliardi: il termine dei parametri: calcolato 0.1745, nel libro 0.175
✓  8 miliardi: il termine dei token: calcolato 0.08399, nel libro 0.084
✓  8 miliardi: loss: calcolato 1.949, nel libro 1.95
✓  Chinchilla: il termine dei parametri: calcolato 0.08349, nel libro 0.083
✓  Chinchilla: il termine dei token: calcolato 0.1632, nel libro 0.163
✓  Chinchilla: loss: calcolato 1.937, nel libro 1.94
✓  quante volte più piccolo («quasi nove»): calcolato 8.75, nel libro 8.75


## Il gradino della misura

*Nel racconto: la diceria delle capacità che compaiono all'improvviso; il tameshiwari.*

In [8]:
for p, atteso in [(0.80, 11), (0.90, 35), (0.95, 60), (0.99, 90)]:
    controlla(f"ogni cifra giusta al {p:.0%}: somma di 10 cifre tutta giusta (%)", p ** 10 * 100, atteso, 0.5)

✓  ogni cifra giusta al 80%: somma di 10 cifre tutta giusta (%): calcolato 10.74, nel libro 11
✓  ogni cifra giusta al 90%: somma di 10 cifre tutta giusta (%): calcolato 34.87, nel libro 35
✓  ogni cifra giusta al 95%: somma di 10 cifre tutta giusta (%): calcolato 59.87, nel libro 60
✓  ogni cifra giusta al 99%: somma di 10 cifre tutta giusta (%): calcolato 90.44, nel libro 90
